# CraterScope 2.0 - training on a free Colab T4
Runtime -> Change runtime type -> **T4 GPU**. Run the cells in order. Everything heavy is saved to Google Drive so a disconnect does not lose work.

Before starting: upload `craterscope2.zip` (the code) to the root of your Google Drive.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code + dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!unzip -q -o /content/drive/MyDrive/craterscope2.zip -d /content/
%cd /content/craterscope2
!pip install -q -r requirements-train.txt
!python config.py

## 2. Datasets (RDD2022 from the official Figshare record)
Set `COUNTRIES`:
* `"India"`: only Indian roads (about 0.5 GB kept).
* `"all"`: India, Japan, Czech, Norway, United_States and China_MotorBike are all used for **training**. China_Drone is skipped because it is top-down.

Validation and test stay India-only either way (`--eval-sources rdd_india` in step 3), so every ablation row is measured on the same Indian-road split.
The first run downloads the ~12.4 GB archive and saves `RDD2022_India.zip` to Drive. Your laptop only needs that file.

In [ ]:
COUNTRIES = "India"          # or "all"
import os, shutil
DRIVE_INDIA = '/content/drive/MyDrive/RDD2022_India.zip'
if COUNTRIES == "India" and os.path.exists(DRIVE_INDIA):
    !mkdir -p downloads/rdd2022 && unzip -q -o {DRIVE_INDIA} -d downloads/rdd2022
else:
    !python get_rdd2022.py --countries {COUNTRIES} --keep-country-zips
    shutil.copy('downloads/RDD2022_India.zip', DRIVE_INDIA)   # the laptop only needs India
    !rm -f downloads/RDD2022_*.zip
!python prep_rdd2022.py --src downloads/rdd2022 --countries {COUNTRIES}

**Optional - HRP4K** (about 8.7 GB, already YOLO format). With `--eval-sources rdd_india` it becomes train-only data, so your laptop does **not** need it.

In [ ]:
# !hf download dronefreak/HRP4K --repo-type dataset --local-dir downloads/hrp4k   # or: huggingface-cli download ...
# !python prep_hrp4k.py --src downloads/hrp4k/data --out datasets/raw/hrp4k

## 3. Frozen split (reuses splits.json from your laptop if present)

In [ ]:
# If you created splits.json on your laptop (it is inside the code zip), rebuild THAT split;
# otherwise create it here once and copy it back to your laptop.
import os
if os.path.exists('splits.json'):
    !python prep_split.py --raw datasets/raw --from-splits splits.json
else:
    !python prep_split.py --raw datasets/raw --eval-sources rdd_india
!cp splits.json /content/drive/MyDrive/craterscope2_splits.json

## 4. Shared gate (tau_abs + logistic presence test)
Row 7's gate is shared by H1 (rows 6-7) and H2 (row 9). Rows 2-4 need their own for the classical-only ablation.

In [ ]:
%%bash
for r in 7 2 3 4; do python fit_gate.py --row $r; done

## 5. H1 verifier: crop dataset, MobileNetV3-small, LBP/HOG SVM, INT8

In [ ]:
!python prep_crops.py --row 7
!python train_verifier.py --kind mnv3 --epochs 20 --batch 256
!python train_verifier.py --kind svm
!python export_models.py --verifier models/verifier_mnv3.onnx

## 6. H2 detector: YOLO11n from COCO weights, 640 px, ~100 epochs
Runs are written to Drive. If Colab disconnects, resume with the commented line.

In [ ]:
!python train_yolo.py --model yolo11n.pt --epochs 100 --batch 32 --project /content/drive/MyDrive/cs_runs
# !python train_yolo.py --model yolo11n.pt --resume /content/drive/MyDrive/cs_runs/yolo11n/weights/last.pt

In [ ]:
# ONNX FP32 + INT8 at 640 and 416 (speed-mAP trade-off)
!python export_models.py --yolo models/yolo11n.pt --imgsz 640
!python export_models.py --yolo models/yolo11n.pt --imgsz 416
# Optional comparison model:
# !python train_yolo.py --model yolov8n.pt --epochs 100 --project /content/drive/MyDrive/cs_runs

## 7. Tune thresholds on VALIDATION only (writes models/tuned.json)

In [ ]:
%%bash
for r in 6 7 8 9; do python evaluate.py --row $r --split val --tune; done

## 8. Package models for the laptop

In [ ]:
!zip -q -r /content/drive/MyDrive/craterscope2_models.zip models splits.json datasets/crops/summary.json
!ls -lh models /content/drive/MyDrive/craterscope2_models.zip